# Save Your Work

Before starting, save this notebook to your Google Drive:
1. Click **File** → **Save a copy in Drive**
2. The copy will open automatically
3. Work in the Google Drive copy from now on

---

# Random Forests

Build ensemble of independent trees for robust predictions.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

## Step 1: Bagging Problem

Single decision tree has high variance — small data changes produce very different trees.

In [ ]:
# Load data
cancer = load_breast_cancer()
X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = pd.Series(cancer.target)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Single unconstrained tree
tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train, y_train)

print(f"Single Decision Tree:")
print(f"  Train accuracy: {tree.score(X_train, y_train):.4f}")
print(f"  Test accuracy:  {tree.score(X_test, y_test):.4f}")
print(f"  Overfitting: {tree.score(X_train, y_train) - tree.score(X_test, y_test):.4f}")

## Step 2: Random Forest Concept

**Bagging (Bootstrap Aggregating):**
1. Create multiple random subsets of training data (sampling with replacement)
2. Train one tree on each subset
3. Average predictions

**Result:** Reduces variance dramatically without increasing bias.

In [ ]:
# Random Forest with default parameters
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

print(f"\nRandom Forest (100 trees):")
print(f"  Train accuracy: {rf.score(X_train, y_train):.4f}")
print(f"  Test accuracy:  {rf.score(X_test, y_test):.4f}")
print(f"  Overfitting: {rf.score(X_train, y_train) - rf.score(X_test, y_test):.4f}")
print(f"\nImprovement over single tree: {(rf.score(X_test, y_test) - tree.score(X_test, y_test)):.4f}")

## Step 3: Effect of n_estimators

In [ ]:
# How many trees do we need?
n_trees_list = [1, 5, 10, 25, 50, 100, 200, 500]
test_scores = []

for n_trees in n_trees_list:
    rf = RandomForestClassifier(n_estimators=n_trees, random_state=42)
    rf.fit(X_train, y_train)
    score = rf.score(X_test, y_test)
    test_scores.append(score)
    print(f"{n_trees:>3} trees: {score:.4f}")

# Plot
plt.figure(figsize=(10, 6))
plt.semilogx(n_trees_list, test_scores, marker='o', linewidth=2)
plt.xlabel('Number of Trees')
plt.ylabel('Test Accuracy')
plt.title('Random Forest: Effect of Number of Trees')
plt.grid(True, alpha=0.3)
plt.show()

print(f"\nAccuracy plateaus around 100 trees. More trees = more memory, no accuracy gain.")

## Step 4: Feature Importance

In [ ]:
# Feature importance
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# Get feature importances
importances = rf.feature_importances_
feature_names = X_train.columns

# Sort and display top 10
indices = np.argsort(importances)[::-1][:10]
print("Top 10 Most Important Features:")
for rank, idx in enumerate(indices, 1):
    print(f"{rank:>2}. {feature_names[idx]:<30} {importances[idx]:.4f}")

## Step 5: Hyperparameter Tuning

In [ ]:
# Tune max_depth
max_depths = [5, 10, 15, 20, 30, None]

print(f"{'Max Depth':<12} {'Train':>8} {'Test':>8} {'Gap':>8}")
print("-" * 37)

for depth in max_depths:
    rf = RandomForestClassifier(
        n_estimators=100, max_depth=depth, random_state=42
    )
    rf.fit(X_train, y_train)
    
    train_score = rf.score(X_train, y_train)
    test_score = rf.score(X_test, y_test)
    gap = train_score - test_score
    
    print(f"{str(depth):<12} {train_score:>8.4f} {test_score:>8.4f} {gap:>8.4f}")

## Step 6: Comparison

In [ ]:
print("\n" + "="*60)
print("COMPARISON: Single Tree vs. Random Forest")
print("="*60)

# Single tree (constrained)
tree = DecisionTreeClassifier(max_depth=15, random_state=42)
tree.fit(X_train, y_train)

# Random forest
rf = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42)
rf.fit(X_train, y_train)

print(f"\nSingle Decision Tree (max_depth=15):")
print(f"  Test accuracy: {tree.score(X_test, y_test):.4f}")
print(f"  Overfitting:   {tree.score(X_train, y_train) - tree.score(X_test, y_test):.4f}")

print(f"\nRandom Forest (100 trees, max_depth=15):")
print(f"  Test accuracy: {rf.score(X_test, y_test):.4f}")
print(f"  Overfitting:   {rf.score(X_train, y_train) - rf.score(X_test, y_test):.4f}")

print(f"\nEnsemble advantage: {(rf.score(X_test, y_test) - tree.score(X_test, y_test)):.4f}")